# EQL und SQL im Vergleich: Metrik-Auswertung

Übersetzt die sechs Evaluationsabfragen mit dem EQL-zu-SQL-Übersetzer und vergleicht jede Abfrage in drei Formen mit den Metriken aus `eql_metrics.py`:

- **EQL**: die Abfrage in EQL, einschließlich der `variable(...)`-Deklarationen
- **Generiertes SQL**: das vom Übersetzer erzeugte SQL
- **Handgeschriebenes SQL**: SQL, wie es eine Person mit Kenntnis des Schemas für dieselbe Frage schreiben würde

Abschnitt 8 führt alle Ergebnisse in einer Zelle zusammen, in derselben Form wie die Tabellen des Evaluationskapitels.

## Setup

In [1]:
%run ./metrics_data_setup.ipynb

from collections import OrderedDict
from IPython.display import Markdown, display
from sqlalchemy import text

from krrood.entity_query_language.factories import entity, variable, an, set_of
from krrood.ormatic.eql_interface import eql_to_sql
import eql_metrics as m

from test.krrood_test.dataset.semantic_world_like_classes import Body, MoveAction

session = create_demo_session()
build_bodies_for_simple_filter(session)

5

## 1. Abfragen übersetzen

Jede Abfrage wird übersetzt und mit `m.analyze(name, eql_source, generated_sql, handwritten_sql)` gemessen. Das PickUpAction-Beispiel braucht die echten pycram-Klassen (ROS2); sein generiertes SQL stammt deshalb aus `eql_metrics.py`.

In [2]:
generated_sql = OrderedDict()

b = variable(type_=Body, domain=[])
generated_sql["Simple filter"] = eql_to_sql(an(entity(b).where(b.size > 5)), session)

move = variable(type_=MoveAction, domain=[])
generated_sql["Transitive attrs"] = eql_to_sql(an(set_of(
    move.robot_x, move.robot_y,
    move.grasp_config.rotate_gripper, move.grasp_config.approach_direction,
)), session)

world_3 = build_container_handle_world()
C = variable(Container, domain=world_3.bodies)
H = variable(Handle, domain=world_3.bodies)
FC = variable(FixedConnection, domain=world_3.connections)
generated_sql["Multi-join (3 vars)"] = eql_to_sql(
    an(set_of(C, H, FC).where(C == FC.parent, H == FC.child)), session
)

world_4 = build_four_variable_world()
C4 = variable(Container, domain=world_4.bodies)
H4 = variable(Handle, domain=world_4.bodies)
FC4 = variable(FixedConnection, domain=world_4.connections)
PC4 = variable(PrismaticConnection, domain=world_4.connections)
generated_sql["Multi-join (4 vars)"] = eql_to_sql(an(set_of(C4, H4, FC4, PC4).where(
    C4 == FC4.parent, H4 == FC4.child, C4 == PC4.child,
)), session)

generated_sql["Complex combined"] = eql_to_sql(
    an(entity(b).where(b.size > 5).ordered_by(b.size).limit(10)), session
)

generated_sql = {name: str(t.sql_query) for name, t in generated_sql.items()}
generated_sql["PickUpAction example"] = m.PICKUP_EXAMPLE_TRANSLATOR_SQL

handwritten_sql = dict(m.HANDWRITTEN_SQL)
handwritten_sql["PickUpAction example"] = m.PICKUP_EXAMPLE_HANDWRITTEN_SQL

queries = [
    m.analyze(name, m.EQL_SOURCES[name], generated_sql[name], handwritten_sql[name])
    for name in generated_sql
]
print(f"{len(queries)} queries translated and measured.")

6 queries translated and measured.


/tmp/ipykernel_19063/495765711.py:33: SAWarning: An alias is being generated automatically against joined entity Mapper[FixedConnectionDAO(FixedConnectionDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  generated_sql = {name: str(t.sql_query) for name, t in generated_sql.items()}
/tmp/ipykernel_19063/495765711.py:33: SAWarning: An alias is being generated automatically against joined entity Mapper[HandleDAO(HandleDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  generated_sql = {name: str(t.sql_query) for name, t in generated_sql.items()}
/tmp/ipykernel_19063/495765711.py:33: SAWarning: A

## 2. Die Abfragen nebeneinander

In [3]:
for r in queries:
    display(Markdown(
        f"### {r['name']}\n\n"
        f"**EQL**\n```python\n{r['eql_source'].strip()}\n```\n\n"
        f"**Generated SQL**\n```sql\n{r['sql_source'].strip()}\n```\n\n"
        f"**Handwritten SQL**\n```sql\n{r['handwritten_source'].strip()}\n```"
    ))

### Simple filter

**EQL**
```python
b = variable(type_=Body, domain=[])
an(entity(b).where(b.size > 5))
```

**Generated SQL**
```sql
SELECT "BodyDAO".database_id, "WorldEntityDAO".database_id AS database_id_1, "SymbolDAO".database_id AS database_id_2, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id 
WHERE "BodyDAO".size > :size_1
```

**Handwritten SQL**
```sql
SELECT database_id, name, size
FROM BodyDAO
WHERE size > 5;
```

### Transitive attrs

**EQL**
```python
move = variable(type_=MoveAction, domain=[])
an(set_of(move.robot_x, move.robot_y, move.grasp_config.rotate_gripper, move.grasp_config.approach_direction))
```

**Generated SQL**
```sql
SELECT "MoveActionDAO".robot_x, "MoveActionDAO".robot_y, "GraspConfigDAO_1".rotate_gripper, "GraspConfigDAO_1".approach_direction 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "MoveActionDAO" ON "MoveActionDAO".database_id = "WorldEntityDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_1" JOIN "WorldEntityDAO" AS "WorldEntityDAO_1" ON "WorldEntityDAO_1".database_id = "SymbolDAO_1".database_id JOIN "GraspConfigDAO" AS "GraspConfigDAO_1" ON "GraspConfigDAO_1".database_id = "WorldEntityDAO_1".database_id) ON "GraspConfigDAO_1".database_id = "MoveActionDAO"._grasp_config_id
```

**Handwritten SQL**
```sql
SELECT m.robot_x, m.robot_y, g.rotate_gripper, g.approach_direction
FROM MoveActionDAO m
JOIN GraspConfigDAO g ON g.database_id = m._grasp_config_id;
```

### Multi-join (3 vars)

**EQL**
```python
C = variable(Container, domain=world.bodies)
H = variable(Handle, domain=world.bodies)
FC = variable(FixedConnection, domain=world.connections)
an(set_of(C, H, FC).where(C == FC.parent, H == FC.child))
```

**Generated SQL**
```sql
SELECT "ContainerDAO".database_id, "BodyDAO".database_id AS database_id_1, "WorldEntityDAO".database_id AS database_id_2, "SymbolDAO".database_id AS database_id_3, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size, "HandleDAO_1".database_id AS database_id_4, "BodyDAO_1".database_id AS database_id_5, "WorldEntityDAO_1".database_id AS database_id_6, "SymbolDAO_1".database_id AS database_id_7, "SymbolDAO_1".polymorphic_type AS polymorphic_type_1, "WorldEntityDAO_1"._world_id AS _world_id_1, "BodyDAO_1".name AS name_1, "BodyDAO_1".size AS size_1, "FixedConnectionDAO_1".database_id AS database_id_8, "ConnectionDAO_1".database_id AS database_id_9, "WorldEntityDAO_2".database_id AS database_id_10, "SymbolDAO_2".database_id AS database_id_11, "SymbolDAO_2".polymorphic_type AS polymorphic_type_2, "WorldEntityDAO_2"._world_id AS _world_id_2, "ConnectionDAO_1"._parent_id, "ConnectionDAO_1"._child_id 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id JOIN "ContainerDAO" ON "ContainerDAO".database_id = "BodyDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_2" JOIN "WorldEntityDAO" AS "WorldEntityDAO_2" ON "WorldEntityDAO_2".database_id = "SymbolDAO_2".database_id JOIN "ConnectionDAO" AS "ConnectionDAO_1" ON "ConnectionDAO_1".database_id = "WorldEntityDAO_2".database_id JOIN "FixedConnectionDAO" AS "FixedConnectionDAO_1" ON "FixedConnectionDAO_1".database_id = "ConnectionDAO_1".database_id) ON "ConnectionDAO_1"._parent_id = "ContainerDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_1" JOIN "WorldEntityDAO" AS "WorldEntityDAO_1" ON "WorldEntityDAO_1".database_id = "SymbolDAO_1".database_id JOIN "BodyDAO" AS "BodyDAO_1" ON "BodyDAO_1".database_id = "WorldEntityDAO_1".database_id JOIN "HandleDAO" AS "HandleDAO_1" ON "HandleDAO_1".database_id = "BodyDAO_1".database_id) ON "ConnectionDAO_1"._child_id = "HandleDAO_1".database_id
```

**Handwritten SQL**
```sql
SELECT c.database_id, c_b.name, c_b.size,
       h.database_id, h_b.name, h_b.size,
       fc.database_id, con._parent_id, con._child_id
FROM ContainerDAO c
JOIN BodyDAO c_b ON c_b.database_id = c.database_id
JOIN ConnectionDAO con ON con._parent_id = c.database_id
JOIN FixedConnectionDAO fc ON fc.database_id = con.database_id
JOIN HandleDAO h ON h.database_id = con._child_id
JOIN BodyDAO h_b ON h_b.database_id = h.database_id;
```

### Multi-join (4 vars)

**EQL**
```python
C = variable(Container, domain=world.bodies)
H = variable(Handle, domain=world.bodies)
FC = variable(FixedConnection, domain=world.connections)
PC = variable(PrismaticConnection, domain=world.connections)
an(set_of(C, H, FC, PC).where(C == FC.parent, H == FC.child, C == PC.child))
```

**Generated SQL**
```sql
SELECT "ContainerDAO".database_id, "BodyDAO".database_id AS database_id_1, "WorldEntityDAO".database_id AS database_id_2, "SymbolDAO".database_id AS database_id_3, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size, "HandleDAO_1".database_id AS database_id_4, "BodyDAO_1".database_id AS database_id_5, "WorldEntityDAO_1".database_id AS database_id_6, "SymbolDAO_1".database_id AS database_id_7, "SymbolDAO_1".polymorphic_type AS polymorphic_type_1, "WorldEntityDAO_1"._world_id AS _world_id_1, "BodyDAO_1".name AS name_1, "BodyDAO_1".size AS size_1, "FixedConnectionDAO_1".database_id AS database_id_8, "ConnectionDAO_1".database_id AS database_id_9, "WorldEntityDAO_2".database_id AS database_id_10, "SymbolDAO_2".database_id AS database_id_11, "SymbolDAO_2".polymorphic_type AS polymorphic_type_2, "WorldEntityDAO_2"._world_id AS _world_id_2, "ConnectionDAO_1"._parent_id, "ConnectionDAO_1"._child_id, "PrismaticConnectionDAO_1".database_id AS database_id_12, "ConnectionDAO_2".database_id AS database_id_13, "WorldEntityDAO_3".database_id AS database_id_14, "SymbolDAO_3".database_id AS database_id_15, "SymbolDAO_3".polymorphic_type AS polymorphic_type_3, "WorldEntityDAO_3"._world_id AS _world_id_3, "ConnectionDAO_2"._parent_id AS _parent_id_1, "ConnectionDAO_2"._child_id AS _child_id_1 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id JOIN "ContainerDAO" ON "ContainerDAO".database_id = "BodyDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_2" JOIN "WorldEntityDAO" AS "WorldEntityDAO_2" ON "WorldEntityDAO_2".database_id = "SymbolDAO_2".database_id JOIN "ConnectionDAO" AS "ConnectionDAO_1" ON "ConnectionDAO_1".database_id = "WorldEntityDAO_2".database_id JOIN "FixedConnectionDAO" AS "FixedConnectionDAO_1" ON "FixedConnectionDAO_1".database_id = "ConnectionDAO_1".database_id) ON "ConnectionDAO_1"._parent_id = "ContainerDAO".database_id JOIN ("SymbolDAO" AS "SymbolDAO_1" JOIN "WorldEntityDAO" AS "WorldEntityDAO_1" ON "WorldEntityDAO_1".database_id = "SymbolDAO_1".database_id JOIN "BodyDAO" AS "BodyDAO_1" ON "BodyDAO_1".database_id = "WorldEntityDAO_1".database_id JOIN "HandleDAO" AS "HandleDAO_1" ON "HandleDAO_1".database_id = "BodyDAO_1".database_id) ON "ConnectionDAO_1"._child_id = "HandleDAO_1".database_id JOIN ("SymbolDAO" AS "SymbolDAO_3" JOIN "WorldEntityDAO" AS "WorldEntityDAO_3" ON "WorldEntityDAO_3".database_id = "SymbolDAO_3".database_id JOIN "ConnectionDAO" AS "ConnectionDAO_2" ON "ConnectionDAO_2".database_id = "WorldEntityDAO_3".database_id JOIN "PrismaticConnectionDAO" AS "PrismaticConnectionDAO_1" ON "PrismaticConnectionDAO_1".database_id = "ConnectionDAO_2".database_id) ON "ConnectionDAO_2"._child_id = "ContainerDAO".database_id
```

**Handwritten SQL**
```sql
SELECT c.database_id, c_b.name, c_b.size,
       h.database_id, h_b.name, h_b.size,
       fc.database_id, con._parent_id, con._child_id,
       pc.database_id, pcon._parent_id, pcon._child_id
FROM ContainerDAO c
JOIN BodyDAO c_b ON c_b.database_id = c.database_id
JOIN ConnectionDAO con ON con._parent_id = c.database_id
JOIN FixedConnectionDAO fc ON fc.database_id = con.database_id
JOIN HandleDAO h ON h.database_id = con._child_id
JOIN BodyDAO h_b ON h_b.database_id = h.database_id
JOIN ConnectionDAO pcon ON pcon._child_id = c.database_id
JOIN PrismaticConnectionDAO pc ON pc.database_id = pcon.database_id;
```

### Complex combined

**EQL**
```python
b = variable(type_=Body, domain=[])
an(entity(b).where(b.size > 5).ordered_by(b.size).limit(10))
```

**Generated SQL**
```sql
SELECT "BodyDAO".database_id, "WorldEntityDAO".database_id AS database_id_1, "SymbolDAO".database_id AS database_id_2, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id 
WHERE "BodyDAO".size > :size_1 ORDER BY "BodyDAO".size
 LIMIT :param_1
```

**Handwritten SQL**
```sql
SELECT database_id, name, size
FROM BodyDAO
WHERE size > 5
ORDER BY size
LIMIT 10;
```

### PickUpAction example

**EQL**
```python
action = variable(type_=PickUpAction, domain=[])
query = an(set_of(
    action.arm,
    action.grasp_description.rotate_gripper,
    action.grasp_description.approach_direction,
))
```

**Generated SQL**
```sql
SELECT "PickUpActionDAO".arm, "GraspDescriptionDAO_1".rotate_gripper, "GraspDescriptionDAO_1".approach_direction
FROM "DesignatorDAO"
JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id
JOIN "PickUpActionDAO" ON "PickUpActionDAO".database_id = "ActionDescriptionDAO".database_id
JOIN "GraspDescriptionDAO" AS "GraspDescriptionDAO_1" ON "GraspDescriptionDAO_1".database_id = "PickUpActionDAO".grasp_description_id
```

**Handwritten SQL**
```sql
SELECT p.arm, g.rotate_gripper, g.approach_direction
FROM PickUpActionDAO p
JOIN GraspDescriptionDAO g ON g.database_id = p.grasp_description_id;
```

## 3. Plausibilitätsprüfungen

Die Asserts prüfen Eigenschaften, die für jede Abfrage gelten müssen. Das handgeschriebene SQL wird nur gemessen, nicht erzeugt. Deshalb wird jede handgeschriebene Abfrage einmal gegen die Demo-Datenbank ausgeführt, damit falsche Tabellen- oder Spaltennamen auffallen (die PickUpAction-Tabellen sind nicht Teil der Demo-Datenbank).

In [4]:
for r in queries:
    assert r["eql"]["fk_error_opportunities"] == 0, r["name"]
    assert r["sql"]["nesting_depth"] <= r["eql"]["nesting_depth"], r["name"]
    assert r["eql"]["extrinsic_load"] <= r["sql"]["extrinsic_load"], r["name"]
    assert r["eql"]["cognitive_complexity"] <= r["sql"]["cognitive_complexity"], r["name"]
    assert r["eql"]["char_length"] < r["sql"]["char_length"], r["name"]

for name, sql in m.HANDWRITTEN_SQL.items():
    rows = session.execute(text(sql.rstrip(";"))).fetchall()
    print(f"{name:<22} handwritten SQL runs, {len(rows)} row(s)")

print("\nAll checks passed.")

Simple filter          handwritten SQL runs, 3 row(s)
Transitive attrs       handwritten SQL runs, 0 row(s)
Multi-join (3 vars)    handwritten SQL runs, 0 row(s)
Multi-join (4 vars)    handwritten SQL runs, 0 row(s)
Complex combined       handwritten SQL runs, 3 row(s)

All checks passed.


## 4. Alle Metriken je Abfrage

Für jede Abfrage alle Metriken für EQL, generiertes SQL und handgeschriebenes SQL nebeneinander.

In [5]:
for r in queries:
    m.print_analysis(r)

--- Simple filter ---
EQL:
  b = variable(type_=Body, domain=[])
  an(entity(b).where(b.size > 5))
SQL (translator): SELECT "BodyDAO".database_id, "WorldEntityDAO".database_id AS database_id_1, "SymbolDAO".database_id AS database_id_2, "SymbolDAO".polymorphic_type, "WorldEntityDAO"._world_id, "BodyDAO".name, "BodyDAO".size 
FROM "SymbolDAO" JOIN "WorldEntityDAO" ON "WorldEntityDAO".database_id = "SymbolDAO".database_id JOIN "BodyDAO" ON "BodyDAO".database_id = "WorldEntityDAO".database_id 
WHERE "BodyDAO".size > :size_1
SQL (handwritten): SELECT database_id, name, size
FROM BodyDAO
WHERE size > 5;

                              EQL        SQL  Handwritten
Char length                    61        377           50
Technical noise                 6         33            5
Domain transparency         0.455      0.154        0.375
Nesting depth                   2          0            0
Halstead volume             44.97     217.13        36.54
FK error opportunities          0          2  

## 5. Skalierung

Eine synthetische Kette aus k Containern, verbunden über k − 1 feste Verbindungen, für k = 2 bis 6 (3 bis 11 Variablen). Gemessen werden nur EQL und generiertes SQL; für diese Reihe gibt es kein handgeschriebenes SQL. Die EQL-Texte enthalten die `variable(...)`-Deklarationen (`m.eql_source_for_chain`). Die Tabelle steht in Abschnitt 8.

In [6]:
scaling_rows = []
for k in range(2, 7):
    world_k = build_container_chain(k)
    containers = [variable(Container, domain=world_k.bodies) for _ in range(k)]
    connections = [variable(FixedConnection, domain=world_k.connections) for _ in range(k - 1)]
    conditions = []
    for i in range(k - 1):
        conditions.append(containers[i] == connections[i].parent)
        conditions.append(containers[i + 1] == connections[i].child)
    query_k = an(set_of(*containers, *connections).where(*conditions))
    scaling_rows.append({
        "variables": 2 * k - 1,
        "eql_len": m.character_length(m.eql_source_for_chain(k)),
        "sql_len": m.character_length(str(eql_to_sql(query_k, session).sql_query)),
    })

assert all(a["eql_len"] < b["eql_len"] and a["sql_len"] < b["sql_len"]
           for a, b in zip(scaling_rows, scaling_rows[1:]))
print(f"{len(scaling_rows)} scaling points measured.")

5 scaling points measured.


/tmp/ipykernel_19063/2329717022.py:14: SAWarning: An alias is being generated automatically against joined entity Mapper[FixedConnectionDAO(FixedConnectionDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  "sql_len": m.character_length(str(eql_to_sql(query_k, session).sql_query)),
/tmp/ipykernel_19063/2329717022.py:14: SAWarning: An alias is being generated automatically against joined entity Mapper[ContainerDAO(ContainerDAO)] due to overlapping tables.  This is a legacy pattern which may be deprecated in a later release.  Use the aliased(<entity>, flat=True) construct explicitly, see the linked example. (Background on this warning at: https://sqlalche.me/e/20/xaj2)
  "sql_len": m.character_length(str(eql_to_sql(query_k, session).sql_query)),


## 6. Feature-Tabelle

In [7]:
m.print_feature_table()

EQL feature coverage:
Feature                                                     Status
------------------------------------------------------------------
SELECT specific columns (set_of)                               yes
SELECT full entity (entity)                                    yes
WHERE / filter conditions                                      yes
Equality joins across variables                                yes
Transitive attribute access (a.b.c)                            yes
ORDER BY (ordered_by)                                          yes
LIMIT                                                          yes
DISTINCT                                                       yes
AND / OR (and_, or_)                                           yes
NOT / IN (not_, in_)                                           yes
GROUP BY (grouped_by)                                          yes
HAVING (having)                                                yes
CASE WHEN (case_when)                   

## 7. Symmetrische und naive Domänen-Transparenz

Domänen-Transparenz des PickUpAction-Beispiels, einmal mit EQLs eigenen DSL-Wörtern als Infrastruktur (symmetrisch, so in allen Tabellen) und einmal ohne (naiv).

In [8]:
print(f"{'':8} {'naive':>8} {'symmetric':>10}")
print(f"{'EQL':<8} {m.domain_transparency_naive(m.PICKUP_EXAMPLE_EQL):>8.3f} "
      f"{m.domain_transparency(m.PICKUP_EXAMPLE_EQL, True):>10.3f}")
print(f"{'Gen SQL':<8} {m.domain_transparency_naive(m.PICKUP_EXAMPLE_TRANSLATOR_SQL):>8.3f} "
      f"{m.domain_transparency(m.PICKUP_EXAMPLE_TRANSLATOR_SQL, False):>10.3f}")

            naive  symmetric
EQL         1.000      0.688
Gen SQL     0.094      0.219


## 8. Gesamtübersicht

Alle Ergebnisse an einer Stelle: die Tabellen des Evaluationskapitels (Spalten in derselben Reihenfolge), die Skalierung, Gleichstände bei der kognitiven Komplexität und die Zählung der Feature-Tabelle.

In [9]:
FORMS = ("eql", "sql", "handwritten")


def print_table(title, columns, values_per_query):
    widths = [max(len(c), 8) for c in columns]
    print(f"\n=== {title} ===")
    print(f"{'Query':<22} " + " ".join(c.rjust(w) for c, w in zip(columns, widths)))
    print("-" * (23 + sum(w + 1 for w in widths)))
    for r in queries:
        values = [str(v) for v in values_per_query(r)]
        print(f"{r['name']:<22} " + " ".join(v.rjust(w) for v, w in zip(values, widths)))


def decomposition(r):
    e, g, h = (r[f]["char_length"] for f in FORMS)
    return [g - e, g - h, h - e, f"{round((g - h) / (g - e) * 100)} %"]


print_table(
    "Cognitive complexity and foreign-key error opportunities",
    ["EQL CC", "Gen CC", "Hand CC", "EQL FK", "Gen FK", "Hand FK"],
    lambda r: [r[f]["cognitive_complexity"] for f in FORMS]
            + [r[f]["fk_error_opportunities"] for f in FORMS],
)
print_table(
    "Character length and ratios",
    ["EQL len", "Gen len", "Hand len", "Gen/EQL", "Hand/EQL", "Gen/Hand"],
    lambda r: [r[f]["char_length"] for f in FORMS] + [
        f"{r['shared']['abstraction_ratio']:.1f}x",
        f"{r['shared']['abstraction_ratio_handwritten']:.1f}x",
        f"{r['shared']['translator_overhead_ratio']:.2f}x",
    ],
)
print_table(
    "Decomposition of the length gap Gen - EQL",
    ["Gen-EQL", "Translator (Gen-Hand)", "Language (Hand-EQL)", "Translator share"],
    decomposition,
)
print_table(
    "Technical noise and domain transparency",
    ["EQL noise", "Gen noise", "Hand noise", "EQL transp", "Gen transp", "Hand transp"],
    lambda r: [r[f]["technical_noise"] for f in FORMS]
            + [f"{r[f]['domain_transparency']:.3f}" for f in FORMS],
)
print_table(
    "Nesting depth, Halstead volume, extrinsic load",
    ["EQL nest", "Gen nest", "Hand nest", "EQL Hals", "Gen Hals", "Hand Hals",
     "EQL ext", "Gen ext", "Hand ext"],
    lambda r: [r[f]["nesting_depth"] for f in FORMS]
            + [f"{r[f]['halstead_volume']:.2f}" for f in FORMS]
            + [r[f]["extrinsic_load"] for f in FORMS],
)

print("\n=== Scaling of the character length ===")
print(f"{'Variables':>9} {'EQL len':>8} {'Gen len':>8} {'Gen/EQL':>8} "
      f"{'EQL growth':>11} {'Gen growth':>11} {'EQL +chars':>11} {'Gen +chars':>11}")
print("-" * 86)
previous = None
for row in scaling_rows:
    e, g = row["eql_len"], row["sql_len"]
    growth = ["--", "--", "--", "--"] if previous is None else [
        f"{e / previous[0]:.2f}x", f"{g / previous[1]:.2f}x", e - previous[0], g - previous[1],
    ]
    print(f"{row['variables']:>9} {e:>8} {g:>8} {g / e:>7.1f}x "
          + " ".join(f"{str(v):>11}" for v in growth))
    previous = (e, g)

print("\n=== Ties ===")
for form, label in (("eql", "EQL"), ("handwritten", "Hand")):
    for i, r1 in enumerate(queries):
        for r2 in queries[i + 1:]:
            if r1[form]["cognitive_complexity"] == r2[form]["cognitive_complexity"]:
                print(f"{label} CC: {r1['name']} = {r2['name']} = {r1[form]['cognitive_complexity']}")
for r in queries:
    if r["eql"]["cognitive_complexity"] == r["handwritten"]["cognitive_complexity"]:
        print(f"EQL CC = Hand CC: {r['name']} ({r['eql']['cognitive_complexity']})")

c = m.feature_coverage_counts()["fine_grained"]
print("\n=== Feature table ===")
print(f"{c['supported']}/{c['total']} supported ({c['percent_supported_only']}%), "
      f"{c['partial']} partial ({c['percent_supported_or_partial']}% incl. partial)")


=== Cognitive complexity and foreign-key error opportunities ===
Query                    EQL CC   Gen CC  Hand CC   EQL FK   Gen FK  Hand FK
-----------------------------------------------------------------------------
Simple filter                 3        5        3        0        2        0
Transitive attrs              1        8        2        0        5        1
Multi-join (3 vars)           4       18        6        0       11        5
Multi-join (4 vars)           5       25        8        0       15        7
Complex combined              3        5        3        0        2        0
PickUpAction example          1        4        2        0        3        1

=== Character length and ratios ===
Query                   EQL len  Gen len Hand len  Gen/EQL Hand/EQL Gen/Hand
-----------------------------------------------------------------------------
Simple filter                61      377       50     6.2x     0.8x    7.54x
Transitive attrs            148      589      13